#### Train model GIÁ CUỐI (dự đoán trực tiếp) — 3 thuật toán

Dự đoán thẳng `target_shown_price` (giá cuối, đã gồm surge) — **Hướng 1**, dùng để đối chiếu với Hướng Hybrid.

**So sanh 3 thuat toan** tren cung 1 bo feature: HistGradientBoosting (sklearn), LightGBM, XGBoost.
Train **theo tung thang** (`evaluation_month`) — khong gop du lieu nhieu thang (lich su gia doi thu
reset theo thang). Cac buoc: (1) model la gi · (2) setup ban dau · (3) qua trinh huan luyen ·
(4) ket qua so bo · (5) luu model + du doan cho evaluation/.

**1. Model huấn luyện là gì**

**Target:** `target_shown_price` (VND), log-transform vì phân phối lệch phải.

Cả 3 model đều là gradient boosting cây quyết định — họ thuật toán tốt nhất cho dữ liệu dạng bảng (tabular), xử lý categorical + NaN native, train nhanh trên CPU.

In [1]:
import warnings, time, sys
from pathlib import Path
sys.path.insert(0, "..")
import numpy as np, pandas as pd
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from _common_train import CAT, D_NUM, dat_categories, prep, ALGOS, metrics
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

PREP = Path("../../data/hcm_train_ready.parquet")
assert PREP.exists(), "Chua co hcm_train_ready.parquet -> chay chuan_bi_du_lieu.ipynb truoc!"
COLS = list(dict.fromkeys(CAT + D_NUM + ["target_shown_price", "persistence_prediction",
        "evaluation_month", "split", "requested_lag_minutes"]))
df = pd.read_parquet(PREP, columns=COLS)

# BAT BUOC: co dinh danh muc categorical tren df DAY DU truoc khi chia train/test
# (neu de prep() tu cast tung tap con, train/test co the co ma so khac nhau -> du doan sai)
df = dat_categories(df)
print(f"Nap {len(df):,} dong x {len(COLS)} cot (chi nap cot can dung, do RAM)")

Nap 6,897,051 dong x 19 cot (chi nap cot can dung, do RAM)


**2. Setup ban đầu (siêu tham số 3 thuật toán)**

| Thuat toan | Sieu tham so chinh | Y nghia |
|---|---|---|
| **HistGB** (sklearn) | max_iter=500, learning_rate=0.05, l2=1.0, early_stopping | Cay boosting histogram, xu ly categorical/NaN native, on dinh |
| **LightGBM** | n_estimators=800, learning_rate=0.03, num_leaves=63, subsample=0.8 | Cay boosting leaf-wise, thuong nhanh hon tren du lieu lon |
| **XGBoost** | n_estimators=800, learning_rate=0.03, max_depth=7, tree_method="hist" | Cay boosting level-wise, enable_categorical=True |

Ca 3 dung chung mot bo feature va cach xu ly categorical (dtype "category") -> so sanh cong bang.

In [2]:
TARGET = "target_shown_price"
NUM = D_NUM
LOG = True  # gia lech phai -> log-target
FEATS = CAT + NUM
print(f"Target={TARGET} | {len(FEATS)} feature | log-target={LOG}")
for algo, tao in ALGOS.items():
    print(f"  [{algo}] {tao()}")

Target=target_shown_price | 14 feature | log-target=True
  [HistGB] HistGradientBoostingRegressor(categorical_features=['service_name',
                                                    'pickup_location_name',
                                                    'dropoff_location_name',
                                                    'weather_main'],
                              early_stopping=True, l2_regularization=1.0,
                              learning_rate=0.05, max_iter=500,
                              n_iter_no_change=20, random_state=42)
  [LightGBM] LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, n_estimators=800,
              num_leaves=63, random_state=42, reg_lambda=1.0, subsample=0.8,
              verbose=-1)
  [XGBoost] XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=20,
             enable_categorical=True, ev

**3. Quá trình huấn luyện (theo từng tháng, 3 cell riêng — mỗi thuật toán 1 cell)**

Với mỗi thuật toán, với mỗi tháng: train trên `split=train`, dự đoán trên `split=test` của cùng tháng.
Để in được **loss train/validation**, LightGBM/XGBoost tách riêng 10% dữ liệu train làm validation nội
bộ (giống cơ chế `validation_fraction=0.1` có sẵn của HistGB) — chỉ dùng để theo dõi/early-stop, không
đụng vào `split=test`.

In [3]:
models = {algo: {} for algo in ALGOS}
tests = {algo: {} for algo in ALGOS}
thangs = sorted(df.evaluation_month.unique())
print("Train theo thang:", thangs)

Train theo thang: ['2026-01', '2026-02', '2026-03']


**3a. HistGB**

`train_score_`/`validation_score_` là điểm nội bộ sklearn (early_stopping tự tách 10% train) — loss = -score.

In [4]:
print("=== HistGB ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    m = ALGOS["HistGB"]()
    ytr = np.log(tr[TARGET]) if LOG else tr[TARGET]
    m.fit(prep(tr, NUM), ytr)
    pred = m.predict(prep(te, NUM))
    te["pred"] = np.exp(pred) if LOG else pred
    models["HistGB"][th] = m; tests["HistGB"][th] = te
    tr_loss = -m.train_score_[-1]; val_loss = -m.validation_score_[-1]
    print(f"  [{th}] train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  so_cay={m.n_iter_}  | {time.time()-t0:.1f}s")

=== HistGB ===
  [2026-01] train_loss=0.0178  val_loss=0.0178  so_cay=500  | 23.5s
  [2026-02] train_loss=0.0178  val_loss=0.0180  so_cay=500  | 25.0s
  [2026-03] train_loss=0.0178  val_loss=0.0181  so_cay=500  | 29.4s


**3b. LightGBM**

Tách riêng 10% train làm validation (`train_test_split`), theo dõi RMSE train/valid, dừng sớm nếu valid không cải thiện sau 20 vòng.

In [5]:
print("=== LightGBM ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    ytr_full = np.log(tr[TARGET]) if LOG else tr[TARGET]
    Xtr, Xval, ytr, yval = train_test_split(prep(tr, NUM), ytr_full, test_size=0.1, random_state=42)
    m = ALGOS["LightGBM"]()
    m.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xval, yval)], eval_names=["train", "valid"],
          eval_metric="rmse", callbacks=[lgb.early_stopping(20, verbose=False)])
    pred = m.predict(prep(te, NUM))
    te["pred"] = np.exp(pred) if LOG else pred
    models["LightGBM"][th] = m; tests["LightGBM"][th] = te
    tr_loss = m.evals_result_["train"]["rmse"][-1]; val_loss = m.evals_result_["valid"]["rmse"][-1]
    print(f"  [{th}] train_rmse={tr_loss:.4f}  val_rmse={val_loss:.4f}  so_cay={m.best_iteration_}  | {time.time()-t0:.1f}s")

=== LightGBM ===
  [2026-01] train_rmse=0.1872  val_rmse=0.1886  so_cay=800  | 24.8s
  [2026-02] train_rmse=0.1873  val_rmse=0.1881  so_cay=800  | 20.8s
  [2026-03] train_rmse=0.1873  val_rmse=0.1893  so_cay=800  | 20.5s


**3c. XGBoost**

Tương tự LightGBM: 10% validation nội bộ, RMSE train/valid, `early_stopping_rounds=20` (đặt trong `_common_train.py`).

In [6]:
print("=== XGBoost ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    ytr_full = np.log(tr[TARGET]) if LOG else tr[TARGET]
    Xtr, Xval, ytr, yval = train_test_split(prep(tr, NUM), ytr_full, test_size=0.1, random_state=42)
    m = ALGOS["XGBoost"]()
    m.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xval, yval)], verbose=False)
    pred = m.predict(prep(te, NUM))
    te["pred"] = np.exp(pred) if LOG else pred
    models["XGBoost"][th] = m; tests["XGBoost"][th] = te
    ev = m.evals_result()
    tr_loss = ev["validation_0"]["rmse"][-1]; val_loss = ev["validation_1"]["rmse"][-1]
    print(f"  [{th}] train_rmse={tr_loss:.4f}  val_rmse={val_loss:.4f}  so_cay={m.best_iteration}  | {time.time()-t0:.1f}s")

=== XGBoost ===
  [2026-01] train_rmse=0.1861  val_rmse=0.1882  so_cay=799  | 41.6s
  [2026-02] train_rmse=0.1861  val_rmse=0.1877  so_cay=799  | 42.1s
  [2026-03] train_rmse=0.1862  val_rmse=0.1889  so_cay=799  | 42.6s


**Gộp kết quả 3 thuật toán**

In [7]:
allte = {algo: pd.concat(tests[algo].values()) for algo in ALGOS}
print("Da train xong ca 3 thuat toan.")

Da train xong ca 3 thuat toan.


**4. Kết quả sơ bộ — so sánh 3 thuật toán**

Gộp dự đoán test (3 tháng) cho từng thuật toán, tính chỉ số. In cả bảng tổng gộp lẫn từng test-set nhỏ theo tháng.

In [8]:
allte = {algo: pd.concat(tests[algo].values()) for algo in ALGOS}

rows = []
for algo in ALGOS:
    d = allte[algo]
    rows.append({"Thuat toan": algo, "Test-set": "TAT CA", "n": len(d), **metrics(d[TARGET], d.pred)})
    for th in thangs:
        dt = d[d.evaluation_month==th]
        rows.append({"Thuat toan": algo, "Test-set": th, "n": len(dt), **metrics(dt[TARGET], dt.pred)})
bang = pd.DataFrame(rows).round(2)
print("SO SANH 3 THUAT TOAN (tong gop + tung test-set nho theo thang):")
display(bang[bang["Test-set"]=="TAT CA"])
print("\nChi tiet tung test-set nho:")
display(bang[bang["Test-set"]!="TAT CA"])

SO SANH 3 THUAT TOAN (tong gop + tung test-set nho theo thang):


,Thuat toan,Test-set,n,MAE,RMSE,R2,MAPE
0,HistGB,TAT CA,864360,18834.01,25644.75,0.70,15.36
4,LightGBM,TAT CA,864360,18809.03,25607.87,0.71,15.34
8,XGBoost,TAT CA,864360,18806.72,25602.59,0.71,15.34



Chi tiet tung test-set nho:


,Thuat toan,Test-set,n,MAE,RMSE,R2,MAPE
1,HistGB,2026-01,315360,18622.75,25327.14,0.71,15.44
2,HistGB,2026-02,234632,18700.76,25422.74,0.71,15.34
3,HistGB,2026-03,314368,19145.39,26121.93,0.70,15.29
5,LightGBM,2026-01,315360,18605.25,25306.00,0.71,15.42
6,LightGBM,2026-02,234632,18670.77,25372.77,0.71,15.32
7,LightGBM,2026-03,314368,19116.64,26079.23,0.70,15.27
9,XGBoost,2026-01,315360,18590.70,25283.05,0.71,15.41
10,XGBoost,2026-02,234632,18676.58,25372.28,0.71,15.33
11,XGBoost,2026-03,314368,19120.54,26087.65,0.70,15.27


**Đối chiếu baseline**

In [9]:
for algo in ALGOS:
    d = allte[algo]
    per = metrics(d[TARGET], d["persistence_prediction"])["MAE"]
    mae = metrics(d[TARGET], d.pred)["MAE"]
    print(f"{algo:10} MAE model={mae:>8,.0f}  MAE persistence={per:>8,.0f}  cai thien={(1-mae/per)*100:+.1f}%")

HistGB     MAE model=  18,834  MAE persistence=  33,683  cai thien=+44.1%
LightGBM   MAE model=  18,809  MAE persistence=  33,683  cai thien=+44.2%
XGBoost    MAE model=  18,807  MAE persistence=  33,683  cai thien=+44.2%


**5. Lưu model + dự đoán (cho notebook evaluation/ dùng)**

Mỗi thuật toán lưu 1 dict model theo tháng vào `../<TenThuatToan>/gia.joblib`. Dự đoán test gộp lưu ra parquet để notebook evaluation nạp lại không cần train lại.

In [10]:
for algo in ALGOS:
    joblib.dump(models[algo], f"../{algo}/gia.joblib")
    print(f"Da luu ../{algo}/gia.joblib")

pred_out = pd.concat([allte[algo].assign(algo=algo) for algo in ALGOS], ignore_index=True)
Path("../evaluation").mkdir(exist_ok=True)
pred_out.to_parquet("../evaluation/pred_gia.parquet", index=False)
print(f"Da luu ../evaluation/pred_gia.parquet ({len(pred_out):,} dong)")
print("=> Mo evaluation/eval_gia.ipynb de danh gia chi tiet.")

Da luu ../HistGB/gia.joblib
Da luu ../LightGBM/gia.joblib
Da luu ../XGBoost/gia.joblib
Da luu ../evaluation/pred_gia.parquet (2,593,080 dong)
=> Mo evaluation/eval_gia.ipynb de danh gia chi tiet.
